# 02 — Global Aggregation, Indices, and NA Handling (Phase 2)

Concatenates all country partials into a single global table per hazard, then computes the exposure index (geometric mean of normalized log(absolute_exposure) and relative_exposure, scaled 0–10) and exposure class (Fisher-Jenks natural breaks) at global/country/regional scopes. Applies NA rules from `countries_NA_summary.csv`.

## Inputs

- Partial CSVs produced by notebook 01

## References

- `config/countries_NA_summary.csv` — countries to mark Not applicable + reasons
- `regions/UNICEF_REP_REG_GLOBAL.csv` — region/regional grouping mapping

## Outputs

`{output_folder}/global/CSV/{hazard}_{date}.csv` — global merged table with indices and classes. This is the main output of this pipeline, per country files in multiple formats are derived from these results.

## Execution order

Run **after** notebook 01. Ensure that all admin2 units have been correctly process so the indices are consistent.


In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing in GEE.
# Phase 2: calculate normalized hazard exposure indices 
# and classes at the global, regional, and country levels.
# Administrative Level: ADM2.
# Author: Angelly Pugliese, Ph.D.
# Date: May 2026
# ============================================================

For a specific set of countries, the numeric values would be set to NaN (blank), and a column will be added to explain the reason (NA: Not Applicable; ID: Insufficient Data; ND: No Data).

In [ ]:
# ============================================================
# Imports
# ============================================================
import sys
from pathlib import Path

# Anchor everything to the # project root (handoff/) so config/bounds/lib resolve regardless of CWD.
PROJECT_ROOT = Path.cwd().parent 
LIB_DIR = PROJECT_ROOT / "lib"
CONFIG_DIR = PROJECT_ROOT / "config"
BOUNDS_DIR = PROJECT_ROOT / "bounds"
REGIONS_DIR = PROJECT_ROOT / "regions"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import pandas as pd
import ee
import os
from GEE_functions import GEEUtils 
from GEE_functions import Hazard
from indicator_functions import IndicatorUtils

import numpy as np
pd.set_option('display.max_columns', 500)

In [ ]:
# ============================================================
# Authenticate and initialize the Earth Engine library
# Define GEE asset path for UNICEF CCRI data
# ============================================================
ee.Authenticate()
# ee.Initialize(project="unicef-ccri")
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
indicator_utils = IndicatorUtils()

In [ ]:
OUTPUT_FOLDER = "output"
PROCESS_SPECIFIC_COUNTRIES = None

In [ ]:
countries_to_process = utils.get_countries_to_process(PROCESS_SPECIFIC_COUNTRIES)
print(f"Countries to process ({len(countries_to_process)}): {countries_to_process}")

# This variable removes global indicators from the analysis when true, considering that only specific countries will be processed
only_regional = False
only_children = False

In [ ]:
if False: # Set to True to get countries from GEE & print country codes and names by type 
    # Get official country list from GEE asset (could change over time)
    countries_info = ee.FeatureCollection(f"{utils.unicef_data_source_path}/{utils.ADMIN0_ASSET}").select(["ISO3", "ucode", "name", "type"], None, False)
    countries_info_list = countries_info.getInfo()
    countries_df = pd.DataFrame([feature['properties'] for feature in countries_info_list['features']])

    # Order df by type and ucode for easier lookup
    countries_df = countries_df.sort_values(by=['type', 'ucode']).reset_index(drop=True)
    countries_df.to_csv("countries_info.csv", index=True)

    for country_type, group in countries_df.groupby('type'):
        print(f"{country_type}: {sorted(group['ucode'].tolist())}")
        print(f"{country_type}: {sorted(group['name'].tolist())}")
        print("\n")

In [ ]:
# Define the lists for rules
# Only state countries are included in the indices/classes (global, regional, national). 
# Non-state countries are included in the global files with blank values for the indices/classes.
# -----------------------------------------------------------
# The non-official codes, sovereignty unsettled countries, insufficient data countries per hazard, 
# ARE INCLUDED in the global files but blank values are set in all numeric columns, 
# and a status (flag) column is added to indicate the reason for exclusion; 
# The options are: insufficient data, not applicable, or no data.

# State countries only for indices/classes calculation.
state_countries = ['AFG_V1', 'AGO_V1', 'ALB_V1', 'AND_V1', 'ARE_V1', 'ARG_V1', 'ARM_V1', 'ATG_V2', 'AUS_V1', 'AUT_V1', 'AZE_V1', 'BDI_V1', 'BEL_V1', 'BEN_V1', 'BFA_V1', 'BGD_V1', 'BGR_V1', 'BHR_V2', 'BHS_V2', 'BIH_V1', 'BLR_V1', 'BLZ_V1', 'BOL_V1', 'BRA_V1', 'BRB_V2', 'BRN_V1', 'BTN_V1', 'BWA_V1', 'CAF_V1', 'CAN_V1', 'CHE_V1', 'CHL_V1', 'CHN_V1', 'CIV_V1', 'CMR_V1', 'COD_V1', 'COG_V1', 'COL_V1', 'COM_V2', 'CPV_V2', 'CRI_V1', 'CUB_V2', 'CYP_V1', 'CZE_V1', 'DEU_V1', 'DJI_V1', 'DMA_V2', 'DNK_V1', 'DOM_V2', 'DZA_V1', 'ECU_V1', 'EGY_V1', 'ERI_V1', 'ESP_V1', 'EST_V1', 'ETH_V1', 'FIN_V1', 'FJI_V2', 'FRA_V1', 'FSM_V2', 'GAB_V1', 'GBR_V1', 'GEO_V1', 'GHA_V1', 'GIN_V1', 'GMB_V1', 'GNB_V1', 'GNQ_V1', 'GRC_V1', 'GRD_V2', 'GTM_V1', 'GUY_V1', 'HND_V1', 'HRV_V1', 'HTI_V2', 'HUN_V1', 'IDN_V1', 'IND_V1', 'IRL_V1', 'IRN_V1', 'IRQ_V1', 'ISL_V1', 'ISR_V1', 'ITA_V1', 'JAM_V2', 'JOR_V1', 'JPN_V1', 'KAZ_V1', 'KEN_V1', 'KGZ_V1', 'KHM_V1', 'KIR_V2', 'KNA_V2', 'KOR_V1', 'KWT_V1', 'LAO_V1', 'LBN_V1', 'LBR_V1', 'LBY_V1', 'LCA_V2', 'LIE_V1', 'LKA_V1', 'LSO_V1', 'LTU_V1', 'LUX_V1', 'LVA_V1', 'MAR_V1', 'MCO_V1', 'MDA_V1', 'MDG_V1', 'MDV_V2', 'MEX_V1', 'MHL_V2', 'MKD_V1', 'MLI_V1', 'MLT_V1', 'MMR_V1', 'MNE_V1', 'MNG_V1', 'MOZ_V1', 'MRT_V1', 'MUS_V2', 'MWI_V1', 'MYS_V1', 'NAM_V1', 'NER_V1', 'NGA_V1', 'NIC_V1', 'NLD_V1', 'NOR_V1', 'NPL_V1', 'NRU_V2', 'NZL_V1', 'OMN_V1', 'PAK_V1', 'PAN_V1', 'PER_V1', 'PHL_V1', 'PLW_V2', 'PNG_V1', 'POL_V1', 'PRK_V1', 'PRT_V1', 'PRY_V1', 'PSE_V1', 'QAT_V1', 'ROU_V1', 'RUS_V1', 'RWA_V1', 'SAU_V1', 'SDN_V1', 'SEN_V1', 'SGP_V2', 'SLB_V2', 'SLE_V1', 'SLV_V1', 'SMR_V1', 'SOM_V1', 'SRB_V1', 'SSD_V1', 'STP_V2', 'SUR_V1', 'SVK_V1', 'SVN_V1', 'SWE_V1', 'SWZ_V1', 'SYC_V2', 'SYR_V1', 'TCD_V1', 'TGO_V1', 'THA_V1', 'TJK_V1', 'TKM_V1', 'TLS_V1', 'TON_V2', 'TTO_V2', 'TUN_V1', 'TUR_V1', 'TUV_V2', 'TZA_V1', 'UGA_V1', 'UKR_V1', 'URY_V1', 'USA_V1', 'UZB_V1', 'VAT_V1', 'VCT_V2', 'VEN_V1', 'VNM_V1', 'VUT_V2', 'WSM_V2', 'YEM_V1', 'ZAF_V1', 'ZMB_V1', 'ZWE_V1']

# Exclude from indices/classes, set blank values and status flag.

#Status: Not applicable
non_official_codes = ["ALA_V1", "CCK_V1", "CXR_V1", "ESH_V1", "GGY_V1", "JEY_V1", "NFK_V1", "PCN_V1", "SJM_V1"]

# Status: Not applicable
sovereignty_unsettled_countries = ['EGY1_V1', 'SDN1_V1', 'SSD1_V1', 'xAB_V1', 'xAC_V1', 'xAP_V1', 'xJK_V1', 'xJL_V1', 'xPI_V1', 'xRI_V1', 'xSI_V1', 'xSK_V1', 'xSR_V1', 'xxx_V1']

# Status: Insufficient data (for all hazards)
id_countries_insufficient_data = ["NIC_V1", "PSE_V1", "CUB_V2"]  # Nicaragua, State of Palestine, Cuba
id_countries_riverine = ["FJI_V2"] # Fiji
id_countries_coastal_flood = ["CPV_V2","MDV_V2","MUS_V2","NRU_V2","WSM_V2","SYC_V2","TON_V2","TUV_V2"]  # cape Verde, Maldives, Mauritius, Nauru, Samoa, Seychelles, Tonga, Tuvalu
id_countries_fire = ["TUV_V2", "MHL_V2", "MDV_V2", "NRU_V2", "LIE_V1"] # Tuvalu, Marshall Islands, Maldives, Nauru, Liechtenstein
id_countries_air_pollution = ["NRU_V2", "MCO_V1"] # Nauru, Monaco

In [ ]:
# ============================================================
# Load hazard metadata
# ============================================================
hazard_info = pd.read_json(CONFIG_DIR / "hazard_info.json")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}, {hazard.threshold}")

In [ ]:
PROCESS_SPECIFIC_HAZARDS = None

In [ ]:
# Get the list of indicator columns to update
indicator_columns = indicator_utils.get_numeric_columns_for_NA()
print(f"Indicator columns to update: {indicator_columns}")

In [ ]:
# ============================================================
# Global Aggregation & Normalization
# Considering that a specific country or countries can be updated,
# the global aggregation and normalization can be rerun independently.
# ============================================================
COUNTRY_COLUMN_NAME = "adm0_ucode"
REGION_COLUMN_NAME = "Region_Code"
POPULATION_CLASSES = ['total', 'total_female', 'total_male', 'under_18_total', 'under_18_female', 'under_18_male'] if not only_children else ['under_18_total', 'under_18_female', 'under_18_male']
for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue
    try:
        hazard_code = hazard.code
        hazard_name:str = hazard.name
        print(f"\nProcessing hazard: {hazard_name} ({hazard_code})")
        # For the hazard, build the global DF with all countries to calculate global indicators
        country_dfs = []
        for country_ucode in countries_to_process:
            try:
                country_df = pd.read_csv(f"{OUTPUT_FOLDER}/hazards/{hazard_code}/partials/{country_ucode}_{hazard_code}_admin2.csv")
                country_df['status'] = ''
                existing_columns = [col for col in indicator_columns if col in country_df.columns]
                if country_ucode in non_official_codes + sovereignty_unsettled_countries:
                    country_df[indicator_columns] = np.nan
                    country_df['status'] = 'Not Applicable'
                elif country_ucode in id_countries_insufficient_data:
                    country_df[indicator_columns] = np.nan
                    country_df['status'] = 'Insufficient Data'
                # Specific hazard-country rules for insufficient data
                elif hazard_code == 'riverine_flood' and country_ucode in id_countries_riverine:
                    country_df[indicator_columns] = np.nan
                    country_df['status'] = 'Insufficient Data'
                elif hazard_code == 'coastal_flood' and country_ucode in id_countries_coastal_flood:
                    country_df[indicator_columns] = np.nan
                    country_df['status'] = 'Insufficient Data'
                elif hazard_code == 'fire' and country_ucode in id_countries_fire:
                    country_df[indicator_columns] = np.nan
                    country_df['status'] = 'Insufficient Data'
                else:
                    mask = country_df['hazs_mean'].isnull() & country_df['exp_total'].isnull()
                    country_df.loc[mask, 'status'] = 'No Data'
                country_dfs.append(country_df)
            except Exception as e:
                print(f"Error reading country {country_ucode} for hazard {hazard_code}")
        df_global = pd.concat(country_dfs, ignore_index=True)
        init_cols = df_global.columns
        print(f"\n Combined ({len(countries_to_process)}) countries: {len(df_global)} total admin2 units")
        
        # Filter to only state countries for the indices/classes calculations, 
        # keep all countries in the final output with blank values for non-state countries.
        state_mask = df_global[COUNTRY_COLUMN_NAME].isin(state_countries) & (df_global['status'] != 'NA') & (df_global['status'] != 'ID') & (df_global['status'] != 'ND')
        df_global_filtered = df_global.loc[state_mask].copy()
        # ============================================================
        # GLOBAL Indices (gexpi, gexpc, gindex)
        # Exposure indices and classes
        # Hazard index
        # ============================================================
        if not only_regional:
            gexpi_cols = [f"gexpi_{pop_class}" for pop_class in POPULATION_CLASSES]
            gexpc_cols = [f"gexpc_{pop_class}" for pop_class in POPULATION_CLASSES]
            df_global = df_global.assign(**{col: np.nan for col in gexpi_cols + gexpc_cols + ["gindex"]})

            for pop_class in POPULATION_CLASSES:
                df_global_filtered = indicator_utils.compute_exposure_index(
                    df=df_global_filtered,
                    log_aexp_col=f"{pop_class}_log_exposure",
                    rexp_col=f"{pop_class}_relative_exposure",
                    output_col=f"gexpi_{pop_class}",
                )
                df_global_filtered = indicator_utils.compute_exposure_class(
                    df=df_global_filtered,
                    expi_col=f"gexpi_{pop_class}",
                    aexp_col=f"{pop_class}_log_exposure",
                    k=5,
                    output_col=f"gexpc_{pop_class}"
                )
            df_global_filtered = indicator_utils.compute_hazard_index(
                df_global_filtered,
                hazs_max_col="hazs_max",
                output_col="gindex"
            )
            # Update the global DataFrame with the computed indices and classes for state countries, keeping non-state countries with blank values
            df_global.loc[state_mask, gexpi_cols + gexpc_cols + ["gindex"]] = df_global_filtered[gexpi_cols + gexpc_cols + ["gindex"]].values

            print(f"✓ Global-level indices computed")

        # ============================================================
        # Country Indices (cexpi, cexpc, cindex)
        # ============================================================
        cexpi_cols = [f"cexpi_{pop_class}" for pop_class in POPULATION_CLASSES]
        cexpc_cols = [f"cexpc_{pop_class}" for pop_class in POPULATION_CLASSES]
        df_global = df_global.assign(cindex=np.nan, **{col: 0.0 for col in cexpi_cols}, **{col: np.nan for col in cexpc_cols})
            
        for country_code, group_indices in df_global_filtered.groupby(COUNTRY_COLUMN_NAME, dropna=False).groups.items():
            if country_code in state_countries:
                idx = group_indices            
                country_admin2_units_length = len(idx)
                if country_admin2_units_length < 5: print(f"computing classes for country {country_code} ({country_admin2_units_length})")
                
                # Country hazard index
                df_c = df_global_filtered.loc[idx].copy()
                df_c = indicator_utils.compute_hazard_index(df_c, hazs_max_col="hazs_max", output_col="cindex")
                df_global_filtered.loc[idx, "cindex"] = df_c["cindex"]
                
                # Country exposure index (country-level normalization)    
                for pop_class in POPULATION_CLASSES:
                    #df_c = df_global_filtered.loc[idx].copy()
                    df_c = indicator_utils.compute_exposure_index(
                        df_c,
                        log_aexp_col=f"{pop_class}_log_exposure",
                        rexp_col=f"{pop_class}_relative_exposure",
                        output_col=f"cexpi_{pop_class}",
                    )
                    df_global_filtered.loc[idx, f"cexpi_{pop_class}"] = df_c[f"cexpi_{pop_class}"]
                    
                    # Country exposure class
                    if df_c[f"cexpi_{pop_class}"].max() == 0:
                        df_global_filtered.loc[idx, f"cexpc_{pop_class}"] = 0
                    elif country_admin2_units_length <= 2:
                        df_global_filtered.loc[idx, f"cexpc_{pop_class}"] = df_global_filtered.loc[idx, f"gexpc_{pop_class}"].values
                    else:
                        k = min(5, max(3, country_admin2_units_length))
                        df_c = indicator_utils.compute_exposure_class(
                            df_c,
                            expi_col=f"cexpi_{pop_class}",
                            aexp_col=f"{pop_class}_log_exposure",
                            k=k,
                            output_col=f"cexpc_{pop_class}"
                        )
                        df_global_filtered.loc[idx, f"cexpc_{pop_class}"] = df_c[f"cexpc_{pop_class}"]

        # Update the global DataFrame with the computed indices and classes for state countries, keeping non-state countries with blank values
        df_global.loc[state_mask, cexpi_cols + cexpc_cols + ["cindex"]] = df_global_filtered[cexpi_cols + cexpc_cols + ["cindex"]].values
    
        print(f"✓ Country-level indices computed")

        #output_col=f"cexpc_{pop_class}"
        # ============================================================
        # REGION-Level Indices (rexpi, rexpc, rindex)
        # ============================================================
        if not only_regional:
            df_global_has_region = df_global.loc[state_mask & ~(df_global["Region_Code"].isna())].copy()

            rexpi_cols = [f"rexpi_{pop_class}" for pop_class in POPULATION_CLASSES]
            rexpc_cols = [f"rexpc_{pop_class}" for pop_class in POPULATION_CLASSES]
            df_global = df_global.assign(rindex=np.nan, **{col: 0.0 for col in rexpi_cols}, **{col: np.nan for col in rexpc_cols})

            for region_code, group_indices in df_global_has_region.groupby(REGION_COLUMN_NAME, dropna=False).groups.items():
                idx = group_indices
                
                n_region = len(idx)
                if n_region < 5: print(f"computing classes for region {region_code} ({n_region})")
                
                # Region hazard index
                df_r = df_global_has_region.loc[idx].copy()
                df_r = indicator_utils.compute_hazard_index(df_r, hazs_max_col="hazs_max", output_col="rindex")
                df_global_has_region.loc[idx, "rindex"] = df_r["rindex"]
                
                # Region exposure index (region-level normalization)
                for pop_class in POPULATION_CLASSES:
                    #df_r = df_global_has_region.loc[idx].copy()
                    df_r = indicator_utils.compute_exposure_index(
                        df_r,
                        log_aexp_col=f"{pop_class}_log_exposure",
                        rexp_col=f"{pop_class}_relative_exposure",
                        output_col=f"rexpi_{pop_class}",
                    )
                    df_global_has_region.loc[idx, f"rexpi_{pop_class}"] = df_r[f"rexpi_{pop_class}"]
                    
                    # Region exposure class
                    if df_r[f"rexpi_{pop_class}"].max() == 0:
                        df_global_has_region.loc[idx, f"rexpc_{pop_class}"] = 0
                    elif n_region <= 2:
                        df_global_has_region.loc[idx, f"rexpc_{pop_class}"] = df_global_has_region.loc[idx, f"gexpc_{pop_class}"].values
                    else:
                        k = min(5, max(3, n_region))
                        df_r = indicator_utils.compute_exposure_class(
                            df_r,
                            expi_col=f"rexpi_{pop_class}",
                            aexp_col=f"{pop_class}_log_exposure",
                            k=k,
                            output_col=f"rexpc_{pop_class}"
                        )
                        df_global_has_region.loc[idx, f"rexpc_{pop_class}"] = df_r[f"rexpc_{pop_class}"]

            # Update the global DataFrame with the computed indices and classes for state countries, keeping non-state countries with blank values
            mask = state_mask & ~(df_global["Region_Code"].isna())
            df_global.loc[mask, rexpi_cols + rexpc_cols + ["rindex"]] = df_global_has_region[rexpi_cols + rexpc_cols + ["rindex"]].values
            print(f"✓ Region-level indices computed")
        
        # ============================================================
        # Final Formatting & Export
        # ============================================================
        rename_map = indicator_utils.get_rename_map()
        df_global = df_global.rename(columns=rename_map)
        columns = df_global.columns

        df_global = indicator_utils.round_cols(df_global)
        
        display_cols = indicator_utils.get_all_indicator_columns()
        
        if only_regional: display_cols = indicator_utils.remove_non_country_level(display_cols)
        if only_children: display_cols = indicator_utils.remove_non_children_indicators(display_cols)
        
        df_global = df_global[display_cols]

        print(f"Finished calculating complete indicator DataFrame")
        # create folder if it dows not exist
        os.makedirs(F"{OUTPUT_FOLDER}/global_NA/CSV", exist_ok=True)
        df_global.to_csv(f"{OUTPUT_FOLDER}/global_NA/CSV/{hazard_code}_{utils.prod_date}.csv", index=False)
    except Exception as e:
        print(f"Error processing hazard {hazard_name} ({hazard_code}): {str(e)}")